# Practical Work 2 — Homography & Perspective Transformation
**Course:** Computer Vision | **Professor:** Slimane Larabi, USTHB

---

## What is a Homography?

A **homography** is a mathematical transformation that maps points from one plane to another using a **3x3 matrix**.

In simple terms: given 4 points in an image (source), it computes a matrix that tells you **where each pixel should go** to match 4 other points (destination).

$$H = \begin{pmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33} \end{pmatrix}$$

A point $(x, y)$ in the source image is mapped to $(x', y')$ in the destination image by:

$$\begin{pmatrix} x' \\ y' \\ 1 \end{pmatrix} = H \cdot \begin{pmatrix} x \\ y \\ 1 \end{pmatrix}$$

**Real world use cases:**
- Document scanning (flatten a photo of a page)
- Augmented Reality (place virtual objects on surfaces)
- Panorama stitching
- Correcting camera distortion

---

## Key OpenCV Functions

### 1. `cv2.findHomography(srcPoints, dstPoints)`
Computes the 3x3 homography matrix from two sets of 4 corresponding points.

```python
H, mask = cv2.findHomography(srcPoints, dstPoints, method, ransacReprojThreshold)
```

| Parameter | Description |
|-----------|-------------|
| `srcPoints` | 4 points in the **source** image (where you clicked) |
| `dstPoints` | 4 corresponding points in the **destination** image (the rectangle corners) |
| `method` | `0` = least squares (default), `cv2.RANSAC` = robust (ignores outliers) |
| `ransacReprojThreshold` | Max pixel error allowed (used with RANSAC, usually 3-10) |
| **Returns** | `H` = the 3x3 matrix, `mask` = which points were used |

**Important:** Points must be given in the **same order** in both arrays. If you click top-left first in src, the first point in dst must also be top-left.

---

### 2. `cv2.warpPerspective(src, H, dsize)`
Applies the homography matrix to **warp the entire image**.

```python
result = cv2.warpPerspective(src, H, (width, height))
```

| Parameter | Description |
|-----------|-------------|
| `src` | The input image to be warped |
| `H` | The 3x3 homography matrix from `findHomography` |
| `dsize` | Size of the output image as `(width, height)` |
| **Returns** | The warped/transformed image |

For every pixel in the output, OpenCV looks back through H to find which pixel in the input it came from.

---

### 3. `cv2.setMouseCallback(windowName, callback)`
Registers a function to be called automatically on mouse events in an OpenCV window.

```python
cv2.setMouseCallback("window_name", myFunction)
```

The callback function must have this exact signature:
```python
def myFunction(event, x, y, flags, param):
    # event: what happened (click, move, etc.)
    # x, y:  pixel coordinates of the mouse
    # flags: extra state (shift held, ctrl held, etc.)
    # param: optional extra data you can pass
```

**Common event types:**

| Event | Meaning |
|-------|---------|
| `cv2.EVENT_LBUTTONDOWN` | Left mouse button clicked |
| `cv2.EVENT_RBUTTONDOWN` | Right mouse button clicked |
| `cv2.EVENT_MOUSEMOVE` | Mouse moved |
| `cv2.EVENT_LBUTTONUP` | Left button released |

---

### 4. `cv2.circle(img, center, radius, color, thickness, lineType)`
Draws a circle on an image.

```python
cv2.circle(img, (x, y), 3, (0, 255, 255), 5, cv2.LINE_AA)
#                         ^radius  ^color    ^thickness  ^antialiased
#                                  BGR format: yellow
```

- `cv2.LINE_AA` = Anti-Aliased = smooth edges (no jagged pixels)
- Color is in **BGR** format (not RGB): `(0, 255, 255)` = Yellow

---

## Coordinate System in OpenCV

```
(0,0) -------- x increases -------->
  |
  |   Image pixels
  |
  y increases
  |
  v
```

The **top-left corner** is `(0, 0)`. The corners of an image of size `(width, height)` are:
- Top-left: `(0, 0)`
- Top-right: `(width-1, 0)`
- Bottom-right: `(width-1, height-1)`
- Bottom-left: `(0, height-1)`

**Note from practice:** When clicking the 4 corners of the book, they must be clicked **clockwise starting from the top-left** so the order matches `pts_dst`. Clicking in the wrong order will result in a mirrored or flipped output.

---

## Exercise 1 — Perspective Correction (homography1.py)

**Goal:** Take a photo of a book taken at an angle and correct it into a flat, front-facing rectangle.

**How it works:**
1. Load the image
2. Click 4 corners of the object (clockwise from top-left)
3. Those 4 clicks become `pts_src`
4. The 4 corners of the output rectangle become `pts_dst` (always fixed)
5. `findHomography` computes the transformation matrix
6. `warpPerspective` applies it to the full image

```
pts_src (your clicks)          pts_dst (fixed rectangle)

  *---------*                    *---------*
  |         |        H           |         |
  |  book   |  ---------->       | straight|
  |  tilted |                    |  book   |
  *---------*                    *---------*
```

Click order: Top-left, Top-right, Bottom-right, Bottom-left

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import sys

### The Mouse Handler Function

This function is **automatically called by OpenCV** every time you interact with the mouse on the window.
It is not called manually — it is registered with `cv2.setMouseCallback()` and OpenCV handles the invocation.

- `global img_temp, pts_src` — required to modify variables defined in the outer scope
- Only reacts to left clicks (`EVENT_LBUTTONDOWN`)
- Draws a yellow dot at the click position as visual feedback
- Saves the point into `pts_src`, limited to a maximum of 4 points

In [ ]:
# -------------------------------------------------------
# MOUSE HANDLER
# Called automatically by OpenCV on each mouse event.
# Signature must always be: (event, x, y, flags, param)
# -------------------------------------------------------
def mouseHandler(event, x, y, flags, param):
    global img_temp, pts_src

    # Only act on left mouse button clicks
    if event == cv2.EVENT_LBUTTONDOWN:

        # Draw a small yellow circle where the user clicked (visual feedback)
        # cv2.circle(image, center, radius, color_BGR, thickness, lineType)
        cv2.circle(img_temp, (x, y), 3, (0, 255, 255), 5, cv2.LINE_AA)
        cv2.imshow("image", img_temp)  # Refresh the window to show the new dot

        # Only collect up to 4 points (the 4 corners of the object)
        if len(pts_src) < 4:
            pts_src = np.append(pts_src, [(x, y)], axis=0)


# -------------------------------------------------------
# LOAD IMAGES
# cv2.imread() returns a NumPy array of shape (height, width, 3)
# Default color format is BGR (not RGB)
# -------------------------------------------------------
imagepath1 = "book1.jpg"
imagepath2 = "book2.jpg"

img1 = cv2.imread(imagepath1)
img2 = cv2.imread(imagepath2)


# -------------------------------------------------------
# DESTINATION IMAGE
# A blank black image of the desired output size.
# np.zeros creates an array filled with 0 (black pixels)
# dtype=uint8 because pixel values are 0-255 (8-bit unsigned integer)
# -------------------------------------------------------
height, width = 400, 600
img_dst = np.zeros((height, width, 3), dtype=np.uint8)


# -------------------------------------------------------
# DESTINATION POINTS (pts_dst)
# The 4 corners of the output rectangle — always fixed.
# Order: Top-left, Top-right, Bottom-right, Bottom-left (clockwise)
# width-1 and height-1 are used because pixel indices start at 0.
# -------------------------------------------------------
pts_dst = np.empty((0, 2))
pts_dst = np.append(pts_dst, [(0, 0)],              axis=0)  # Top-left
pts_dst = np.append(pts_dst, [(width-1, 0)],        axis=0)  # Top-right
pts_dst = np.append(pts_dst, [(width-1, height-1)], axis=0)  # Bottom-right
pts_dst = np.append(pts_dst, [(0, height-1)],       axis=0)  # Bottom-left


# -------------------------------------------------------
# WINDOW SETUP
# cv2.namedWindow creates a named window.
# The value 1 means the window is resizable.
# -------------------------------------------------------
cv2.namedWindow("image", 1)

# img_temp is the image shown in the window.
# The mouse handler draws circles on it directly.
img_temp = img1

# pts_src will be filled by mouse clicks — starts empty
pts_src = np.empty((0, 2))

# Register the mouse handler for the window
cv2.setMouseCallback("image", mouseHandler)

# Show the image and wait for 4 clicks, then press any key to continue
cv2.imshow("image", img_temp)
cv2.waitKey(0)


# -------------------------------------------------------
# COMPUTE HOMOGRAPHY
# pts_src = the 4 points clicked on the book
# pts_dst = the 4 corners of the destination rectangle
# tform   = the resulting 3x3 transformation matrix
# status  = which points were used (relevant with RANSAC)
# -------------------------------------------------------
tform, status = cv2.findHomography(pts_src, pts_dst)


# -------------------------------------------------------
# APPLY PERSPECTIVE WARP
# Applies tform to every pixel in img1.
# Output size is (width, height) — the flat rectangle.
# -------------------------------------------------------
img_dst = cv2.warpPerspective(img1, tform, (width, height))


# -------------------------------------------------------
# DISPLAY AND SAVE THE RESULT
# -------------------------------------------------------
cv2.imshow("image", img_dst)
cv2.imwrite("output.png", img_dst)
cv2.waitKey(0)

---

## Summary — Full Program Flow

```
Load image (book1.jpg)
        |
        v
Open window + register mouse callback
        |
        v
waitKey(0)  ->  click 4 corners of the book  ->  pts_src filled
        |
        v
Press any key to continue
        |
        v
findHomography(pts_src, pts_dst)  ->  3x3 matrix H
        |
        v
warpPerspective(img1, H, (600, 400))  ->  flat corrected image
        |
        v
Display and save result
```

---

## Quick Reference

| Function | Purpose | Key parameters |
|----------|---------|----------------|
| `cv2.imread(path)` | Load image from disk | path as string |
| `cv2.imshow(name, img)` | Display image in window | window name, image array |
| `cv2.waitKey(0)` | Wait for keypress | 0 = infinite wait |
| `cv2.namedWindow(name, flag)` | Create a window | 1 = resizable |
| `cv2.setMouseCallback(name, fn)` | Register mouse function | window name, function |
| `cv2.circle(img, center, r, color, t)` | Draw a circle | center as (x,y), BGR color |
| `cv2.findHomography(src, dst)` | Compute H matrix | 4 source pts, 4 dest pts |
| `cv2.warpPerspective(img, H, size)` | Apply homography | image, H matrix, (w,h) |
| `cv2.imwrite(path, img)` | Save image to disk | path as string |
| `np.zeros((h,w,3), dtype=np.uint8)` | Create black image | height, width, 3 channels |
| `np.append(arr, [(x,y)], axis=0)` | Add point to array | axis=0 adds a row |
| `np.empty((0,2))` | Empty 2D array | 0 rows, 2 columns (x and y) |